# Cuaderno común CR01 — Descarga de datos de reanálisis

**Cambio Climático · Cuaderno común · Ingeniería Ambiental e Ingeniería en Riesgos de Desastres · ESPAM MFL** · Material CC-COM-CR01 · Versión 5.0 · octubre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb)

**Objetivo.** Obtener, solo para su zona (provincia o cantón), las 9 variables diarias y mensuales de dos reanálisis: ERA5 (ECMWF) y CORe (NOAA).

**Producto.** La carpeta `Mi unidad/CC_datos/<zona>/` con los archivos NetCDF de su zona y un inventario. Todas las prácticas, de las dos carreras, trabajan sobre ella.

**Requisito.** Opción A: su **propia** cuenta en el Climate Data Store, con la licencia de ERA5 aceptada, y su **propio** API Token. Opción B: ninguno.

**Cómo trabajar.** En Colab: **Archivo › Guardar una copia en Drive**. Elija sus datos en el formulario de la celda **1** (listas desplegables a la derecha) y ejecute las demás en orden.

<details><summary>Secuencia y variables</summary>


| Orden | Cuaderno común (las dos carreras) | Qué hace | Deja listo en `Mi unidad/CC_datos/<zona>/` |
|---|---|---|---|
| C1 | [`CR01_descarga_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR01_descarga_reanalisis.ipynb) | Descarga ERA5 y CORe solo para su zona (opción A, con su cuenta) o los trae del release del curso (opción B) | Archivos NetCDF e inventario |
| C2 | [`CR02_series_zona`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/comun/reanalisis/CR02_series_zona.ipynb) | Promedia sobre la zona, ponderando cada celda por su área dentro del límite | Series diaria y mensual |

| Carrera | Continúa con | Productos en |
|---|---|---|
| Ingeniería Ambiental | [`IA1_01_climatologia`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_01_climatologia.ipynb) → [`IA1_02_estacion_vs_reanalisis`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-1/IA1_02_estacion_vs_reanalisis.ipynb) | `Mi unidad/CC_Ambiental/practica-1/<zona>/` |
| Ingeniería en Riesgos de Desastres | [`RD1_01_ciclo_anual_enso`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-1/RD1_01_ciclo_anual_enso.ipynb) → [`RD1_02_extremos_eventos`](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/riesgos-desastres/practica-1/RD1_02_extremos_eventos.ipynb) | `Mi unidad/CC_Riesgos/practica-1/<zona>/` |


| Variable | Columna | Unidad |
|---|---|---|
| Temperatura media, máxima y mínima a 2 m | `tmed`, `tmax`, `tmin` | °C |
| Precipitación | `pr` | mm/día · mm/mes |
| Humedad relativa a 2 m | `hr` | % |
| Humedad específica a 2 m | `q` | g/kg |
| Presión atmosférica en superficie | `ps` | hPa |
| Viento a 10 m: componentes, velocidad y dirección de procedencia | `u10`, `v10`, `ws`, `wd` | m/s · ° |

</details>

© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

## 1. Datos del estudiante

In [1]:
# @title Datos del estudiante y zona de trabajo (la misma en todos los cuadernos)
ESTUDIANTE = "Cristian Paliz"  # @param {type:"string"}
NIVEL = "provincia"  # @param ["provincia", "cantón"]
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
CANTON = "Manabí · Portoviejo"  # @param ["Azuay · Camilo Ponce Enriquez", "Azuay · Chordeleg", "Azuay · Cuenca", "Azuay · El Pan", "Azuay · Giron", "Azuay · Guachapala", "Azuay · Gualaceo", "Azuay · Nabon", "Azuay · Oña", "Azuay · Paute", "Azuay · Pucara", "Azuay · San Fernando", "Azuay · Santa Isabel", "Azuay · Sevilla De Oro", "Azuay · Sigsig", "Bolívar · Caluma", "Bolívar · Chillanes", "Bolívar · Chimbo", "Bolívar · Echeandia", "Bolívar · Guaranda", "Bolívar · Las Naves", "Bolívar · San Miguel", "Cañar · Azogues", "Cañar · Biblian", "Cañar · Cañar", "Cañar · Deleg", "Cañar · El Tambo", "Cañar · La Troncal", "Cañar · Suscal", "Carchi · Bolivar", "Carchi · Espejo", "Carchi · Mira", "Carchi · Montufar", "Carchi · San Pedro De Huaca", "Carchi · Tulcan", "Chimborazo · Alausi", "Chimborazo · Chambo", "Chimborazo · Chunchi", "Chimborazo · Colta", "Chimborazo · Cumanda", "Chimborazo · Guamote", "Chimborazo · Guano", "Chimborazo · Pallatanga", "Chimborazo · Penipe", "Chimborazo · Riobamba", "Cotopaxi · La Mana", "Cotopaxi · Latacunga", "Cotopaxi · Pangua", "Cotopaxi · Pujili", "Cotopaxi · Salcedo", "Cotopaxi · Saquisili", "Cotopaxi · Sigchos", "El Oro · Arenillas", "El Oro · Atahualpa", "El Oro · Balsas", "El Oro · Chilla", "El Oro · El Guabo", "El Oro · Huaquillas", "El Oro · Las Lajas", "El Oro · Machala", "El Oro · Marcabeli", "El Oro · Pasaje", "El Oro · Piñas", "El Oro · Portovelo", "El Oro · Santa Rosa", "El Oro · Zaruma", "Esmeraldas · Atacames", "Esmeraldas · Eloy Alfaro", "Esmeraldas · Esmeraldas", "Esmeraldas · La Concordia", "Esmeraldas · Muisne", "Esmeraldas · Quininde", "Esmeraldas · Rioverde", "Esmeraldas · San Lorenzo", "Galápagos · Isabela", "Galápagos · San Cristobal", "Galápagos · Santa Cruz", "Guayas · Alfredo Baquerizo Moreno", "Guayas · Balao", "Guayas · Balzar", "Guayas · Colimes", "Guayas · Crnel. Marcelino Maridueña", "Guayas · Daule", "Guayas · Duran", "Guayas · El Triunfo", "Guayas · Empalme", "Guayas · Gnral. Antonio Elizalde", "Guayas · Guayaquil", "Guayas · Isidro Ayora", "Guayas · Lomas De Sargentillo", "Guayas · Milagro", "Guayas · Naranjal", "Guayas · Naranjito", "Guayas · Nobol", "Guayas · Palestina", "Guayas · Pedro Carbo", "Guayas · Playas", "Guayas · Salitre", "Guayas · Samborondon", "Guayas · San Jacinto De Yaguachi", "Guayas · Santa Lucia", "Guayas · Simon Bolivar", "Imbabura · Antonio Ante", "Imbabura · Cotacachi", "Imbabura · Ibarra", "Imbabura · Otavalo", "Imbabura · Pimampiro", "Imbabura · San Miguel De Urcuqui", "Loja · Calvas", "Loja · Catamayo", "Loja · Celica", "Loja · Chaguarpamba", "Loja · Espindola", "Loja · Gonzanama", "Loja · Loja", "Loja · Macara", "Loja · Olmedo", "Loja · Paltas", "Loja · Pindal", "Loja · Puyango", "Loja · Quilanga", "Loja · Saraguro", "Loja · Sozoranga", "Loja · Zapotillo", "Los Ríos · Baba", "Los Ríos · Babahoyo", "Los Ríos · Buena Fe", "Los Ríos · Mocache", "Los Ríos · Montalvo", "Los Ríos · Palenque", "Los Ríos · Puebloviejo", "Los Ríos · Quevedo", "Los Ríos · Quinsaloma", "Los Ríos · Urdaneta", "Los Ríos · Valencia", "Los Ríos · Ventanas", "Los Ríos · Vinces", "Manabí · 24 De Mayo", "Manabí · Bolivar", "Manabí · Chone", "Manabí · El Carmen", "Manabí · Flavio Alfaro", "Manabí · Jama", "Manabí · Jaramijo", "Manabí · Jipijapa", "Manabí · Junin", "Manabí · Manta", "Manabí · Montecristi", "Manabí · Olmedo", "Manabí · Pajan", "Manabí · Pedernales", "Manabí · Pichincha", "Manabí · Portoviejo", "Manabí · Puerto Lopez", "Manabí · Rocafuerte", "Manabí · San Vicente", "Manabí · Santa Ana", "Manabí · Sucre", "Manabí · Tosagua", "Morona Santiago · Gualaquiza", "Morona Santiago · Huamboya", "Morona Santiago · Limon Indanza", "Morona Santiago · Logroðo", "Morona Santiago · Morona", "Morona Santiago · Pablo Sexto", "Morona Santiago · Palora", "Morona Santiago · San Juan Bosco", "Morona Santiago · Santiago", "Morona Santiago · Sucua", "Morona Santiago · Taisha", "Morona Santiago · Tiwintza", "Napo · Archidona", "Napo · Carlos Julio Arosemena Tola", "Napo · El Chaco", "Napo · Quijos", "Napo · Tena", "Orellana · Aguarico", "Orellana · La Joya De Los Sachas", "Orellana · Loreto", "Orellana · Orellana", "Pastaza · Arajuno", "Pastaza · Mera", "Pastaza · Pastaza", "Pastaza · Santa Clara", "Pichincha · Cayambe", "Pichincha · Mejia", "Pichincha · Pedro Moncayo", "Pichincha · Pedro Vicente Maldonado", "Pichincha · Puerto Quito", "Pichincha · Quito", "Pichincha · Rumiðahui", "Pichincha · San Miguel De Los Bancos", "Santa Elena · La Libertad", "Santa Elena · Salinas", "Santa Elena · Santa Elena", "Santo Domingo de los Tsáchilas · Santo Domingo", "Sucumbíos · Cascales", "Sucumbíos · Cuyabeno", "Sucumbíos · Gonzalo Pizarro", "Sucumbíos · Lago Agrio", "Sucumbíos · Putumayo", "Sucumbíos · Shushufindi", "Sucumbíos · Sucumbios", "Tungurahua · Ambato", "Tungurahua · Baños De Agua Santa", "Tungurahua · Cevallos", "Tungurahua · Mocha", "Tungurahua · Patate", "Tungurahua · Quero", "Tungurahua · San Pedro De Pelileo", "Tungurahua · Santiago De Pillaro", "Tungurahua · Tisaleo", "Zamora Chinchipe · Centinela Del Condor", "Zamora Chinchipe · Chinchipe", "Zamora Chinchipe · El Pangui", "Zamora Chinchipe · Nangaritza", "Zamora Chinchipe · Palanda", "Zamora Chinchipe · Paquisha", "Zamora Chinchipe · Yacuambi", "Zamora Chinchipe · Yantzaza", "Zamora Chinchipe · Zamora"]
# NIVEL = "provincia": se usa PROVINCIA y NO se usa CANTON.
# NIVEL = "cantón": se usa CANTON (la lista está ordenada por provincia) y NO se usa PROVINCIA:
#                   la provincia sale del propio cantón.
if NIVEL == "cantón":
    _provincia_formulario = PROVINCIA
    PROVINCIA, CANTON = CANTON.split(" · ")
    print(f"Zona elegida: cantón {CANTON}, provincia de {PROVINCIA}.")
    if _provincia_formulario != PROVINCIA:
        print(f"  Aviso: el campo PROVINCIA dice {_provincia_formulario}, pero no se usa en el nivel cantón.")
else:
    print(f"Zona elegida: provincia de {PROVINCIA}. El campo CANTON no se usa en el nivel provincia.")
    CANTON = None

ANIO_INI, ANIO_FIN = 1981, 2024   # periodo de datos del curso: no cambiar

# De dónde salen los datos (ver sección 4):
#   "propio"  = opción A: usted los descarga con su cuenta del Climate Data Store (recomendado)
#   "release" = opción B: los trae ya preparados del release del curso (si el docente lo indica)
MODO_DATOS = "release"  # @param ["propio", "release"]
# Solo opción A: hora local en que empieza el día de ERA5 (0 = 00–24 h; 7 = 07–07 h, como el INAMHI)
INICIO_DIA = 7  # @param [0, 7] {type:"raw"}
# Solo opción A: marcado = CORe diario (índices de extremos; varias horas, reanudable); sin marcar = solo mensual (minutos)
CORE_DIARIO = True  # @param {type:"boolean"}

# Credencial PERSONAL del Climate Data Store (solo opción A, ver sección 5). Vacío si la guardó en Secrets.
CDS_TOKEN = "f6e02ffb-34df-43dc-9e7b-f04bd1a5bf71"  # @param {type:"string"}

Zona elegida: provincia de Manabí. El campo CANTON no se usa en el nivel provincia.


## 2. Entorno de trabajo

Instala dos librerías, trae el código del curso desde GitHub (`herramientas/`), conecta su Drive y ubica la zona en la capa oficial de límites.

In [ ]:
!pip -q install cdsapi eccodes

import os, sys, pathlib, subprocess, datetime, platform, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import limites, variables as V
from herramientas.reanalisis import era5, core, release
from herramientas.estaciones import calidad

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(".")

import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, geopandas as gpd
Z = limites.zona(PROVINCIA, CANTON)
GEOM, S = Z["geom"], Z["slug"]
DATOS = BASE / "CC_datos" / S            # datos y series comunes a las dos carreras (CR01, CR02)
RES = BASE / "CC_datos" / S                        # productos de este cuaderno
for c in (DATOS, RES): c.mkdir(parents=True, exist_ok=True)
print(f"Zona: {Z['nombre']} ({Z['tipo']}, DPA {Z['codigo']}, región {Z['region']})")
print(f"Datos en {DATOS} · resultados en {RES}")

## 3. Confirme su zona

La tabla lista las provincias o, si eligió `cantón`, los cantones de su provincia, con su código DPA, superficie y número de celdas de ERA5 (0,25°) que tocan cada uno: más celdas es más tiempo de descarga en la opción A y una serie más representativa. El mapa marca en naranja la zona elegida. **Si no es la que quería, cambie la celda 1 y vuelva a ejecutar desde allí.**

Un cantón con 1 o 2 celdas queda representado por muy pocos puntos del reanálisis: es válido, pero téngalo en cuenta al interpretar.

In [ ]:
print(f"Zona elegida: {Z['nombre']} · {Z['tipo']} · DPA {Z['codigo']} · región {Z['region']}")
display(limites.tabla_zonas(None if NIVEL == "provincia" else Z["provincia"]))
limites.mapa_zonas(Z); plt.show()

## 4. Dos formas de obtener los datos

| | **A. `"propio"`: usted descarga** | **B. `"release"`: el docente ya descargó** |
|---|---|---|
| Quién descarga | Usted, con **su** cuenta del CDS, en **su** Colab | El docente, una vez, para las 24 provincias |
| Dónde quedan | En su Drive, solo su zona | En el release del repositorio; usted trae y recorta su provincia |
| ERA5 | Diario y mensual, con el día que usted elija | Diario y mensual, día 00–24 h |
| CORe | Diario y mensual (`CORE_DIARIO = True`) o solo mensual | Diario y mensual |
| Tiempo | ERA5: ~0,5 a 1 min por celda (sección 6). CORe diario: varias horas; solo mensual: minutos | 1 a 2 minutos |
| Lo que aprende | El proceso completo, reproducible por cualquiera | El análisis |

Las dos opciones dejan **los mismos archivos con los mismos nombres**: el cuaderno común `CR02_series_zona` y las prácticas de cada carrera funcionan igual.

**Por qué CORe diario tarda tanto.** CORe se publica en archivos **globales de 6 horas**: para un día se necesitan 4 archivos, y para 1981–2024 unos 64 000, sin importar el tamaño de su zona. De cada archivo se piden solo los bytes de los 8 campos (~0,4 MB, en lugar de 1,6 MB) y en su Drive queda solo el recorte de la zona, pero el número de solicitudes no se puede reducir. Por eso se guarda mes a mes: puede detenerlo y seguir otro día. Si solo necesita el ciclo anual, use `CORE_DIARIO = False`; para índices de extremos diarios, necesita el diario.

**Por qué ERA5 por celdas.** El conjunto de series por punto del CDS guarda los datos ordenados por lugar: cada celda de 0,25° se entrega completa (44 años, hora a hora) en segundos. Se piden **todas las celdas que tocan la zona**, se pasa de horario a diario en hora local y se vuelve a armar la malla. La provincia queda representada con todas sus celdas, ponderadas en `CR02_series_zona` por la fracción de su área que cae dentro del límite.

## 5. Su credencial personal del Climate Data Store (solo opción A)

**Cada estudiante usa su propia cuenta y su propio API Token.** El CDS pone en cola y limita las solicitudes **por cuenta**: si todo el curso usara una sola, las descargas se bloquearían entre sí y la cuenta podría quedar suspendida.

Dónde está su token: entre a **cds.climate.copernicus.eu**, inicie sesión, abra su perfil (su nombre, arriba a la derecha) y copie el **API Token**. Acepte además las licencias de *ERA5 hourly time-series data on single levels* y de *ERA5 monthly averaged data on single levels* (al final de la pestaña *Download* de cada conjunto).

| Forma | Cómo | Recomendación |
|---|---|---|
| **A. Secrets de Colab** | Ícono de llave a la izquierda › *Add new secret* › nombre `CDSAPI_KEY`, valor: su token › active el acceso para este cuaderno | La más segura: el token no queda escrito en el cuaderno |
| **B. En la celda 1** | Pegue el token entre las comillas de `CDS_TOKEN` | Práctica, pero **no comparta** su copia del cuaderno con el token escrito |

Si no hace ninguna de las dos, la celda siguiente se lo pedirá y no quedará escrito en ningún lado. Muestra solo el inicio y el final del token, para que confirme que es el suyo.

In [ ]:
assert MODO_DATOS in ("propio", "release"), 'MODO_DATOS debe ser "propio" o "release"'
if MODO_DATOS == "propio":
    print("Credencial activa:", era5.credenciales(CDS_TOKEN))
else:
    print("Opción B: no se necesita credencial. Release de datos:", release.ETIQUETA)

## 6. Su zona en las mallas de los dos reanálisis

Cada cuadro es una celda de ERA5 (0,25°, ~28 km) que toca la zona; el color indica qué fracción de su área cae dentro del límite. El rectángulo azul es el recuadro de CORe (~1,9°): sus celdas son unas 50 veces más grandes.

In [ ]:
CELDAS = limites.celdas(GEOM, 0.25)
AREA = limites.recuadro(GEOM, resolucion=0.25, margen_celdas=1)       # ERA5 (terreno)
AREA_CORE = limites.recuadro(GEOM, resolucion=1.875, margen_celdas=1)  # CORe
print(f"ERA5: {len(CELDAS)} celdas tocan {Z['nombre']} (área equivalente a {CELDAS['fraccion'].sum():.1f} celdas completas)")
print("Recuadro CORe [N, O, S, E]:", AREA_CORE)
if MODO_DATOS == "propio":
    print(f"Tiempo estimado de ERA5: {0.5 * len(CELDAS):.0f} a {1.0 * len(CELDAS):.0f} min (depende de la cola del CDS)")

from matplotlib.patches import Rectangle
fig, ax = plt.subplots(figsize=(6, 6))
limites.cargar("provincias").boundary.plot(ax=ax, color="0.75", linewidth=0.5)
for r in CELDAS.itertuples():
    ax.add_patch(Rectangle((r.lon - 0.125, r.lat - 0.125), 0.25, 0.25, facecolor=plt.cm.YlOrBr(r.fraccion), edgecolor="0.4", lw=0.3))
gpd.GeoSeries([GEOM]).boundary.plot(ax=ax, color="k", lw=1)
n, o, s, e = AREA_CORE
ax.plot([o, e, e, o, o], [n, n, s, s, n], "b:", label="Recuadro CORe")
ax.set_xlim(o - 0.3, e + 0.3); ax.set_ylim(s - 0.3, n + 0.3); ax.legend(loc="lower left")
ax.set_title(f"{Z['nombre']}: {len(CELDAS)} celdas de ERA5"); plt.show()

## 7. Descarga

**Opción A.** Primero ERA5, celda por celda: cada celda se guarda en `CC_datos/<zona>/_celdas_era5/` apenas llega. **Si Colab se desconecta, vuelva a ejecutar el cuaderno desde la celda 1: seguirá desde la última celda guardada.** Luego una solicitud pequeña con la altura del terreno del modelo, y por último CORe: el diario se guarda **mes a mes** en `_dias_core/` (4 descargas a la vez, con la estimación del tiempo que falta); el mensual de `CORE_DIARIO = False` son 528 archivos de unos 200 kB.

**Opción B.** Se trae cada archivo de su provincia, se recorta al recuadro de su zona y al periodo, y se borra el archivo provincial.

Cuidado de los servidores en los dos casos: si un archivo ya está en su Drive no se vuelve a pedir; si una solicitud falla se reintenta con espera creciente; nunca se piden datos fuera de su zona.

In [ ]:
anios = range(ANIO_INI, ANIO_FIN + 1)
ARCH = {}
if MODO_DATOS == "propio":
    r = era5.descargar_zona(GEOM, anios, DATOS, S, inicio_dia=INICIO_DIA)
    ARCH["era5_diario"], ARCH["era5_mensual"] = r["diario"], r["mensual"]
    ARCH["era5_terreno"] = era5.descargar_terreno(AREA, DATOS / f"era5_terreno_{S}.nc")
    ARCH["core_diario"] = ARCH["core_mensual"] = None
    try:
        if CORE_DIARIO:   # el mensual se calcula de los diarios, como en ERA5
            r = core.diario_zona(AREA_CORE, anios, DATOS, S)
            ARCH["core_diario"], ARCH["core_mensual"] = r["diario"], r["mensual"]
        else:
            ARCH["core_mensual"] = core.mensual_zona(AREA_CORE, anios, DATOS, S)
    except Exception as err:
        print(f"CORe no disponible ahora ({err}). Vuelva a ejecutar esta celda más tarde: seguirá donde quedó.")
else:
    for fuente, frec in release.PRODUCTOS:
        caja = AREA_CORE if fuente == "core" else AREA
        ARCH[f"{fuente}_{frec}"] = release.obtener(fuente, frec, Z, caja, anios, DATOS)
for k, v in ARCH.items():
    print(f"{k:14s}", "—" if v is None else v.name)

## 8. Inventario y control

Periodo, número de pasos de tiempo, malla, variables y porcentaje de vacíos de cada archivo. El inventario queda junto a los datos: es la evidencia de qué se descargó, de dónde y cuándo.

In [ ]:
filas = []
for k, f in ARCH.items():
    if f is None:
        filas.append({"archivo": k, "estado": "no disponible"}); continue
    ds = era5.abrir_ds(f)
    t = ds["time"].values if "time" in ds.dims else [np.datetime64("NaT", "ns")]
    ref = "tmed" if "tmed" in ds else list(ds.data_vars)[0]
    filas.append({"archivo": f.name, "estado": "ok", "variables": ", ".join(ds.data_vars),
                  "desde": str(np.min(t))[:10], "hasta": str(np.max(t))[:10], "pasos": int(ds.sizes.get("time", 1)),
                  "malla": f"{ds.sizes['lat']}×{ds.sizes['lon']}", "MB": round(f.stat().st_size / 1e6, 2),
                  "celdas_vacías_%": round(100 * float(ds[ref].isnull().all("time").mean()), 1) if "time" in ds.dims else 0.0})
INVENTARIO = pd.DataFrame(filas)
display(INVENTARIO)
print(f"Espacio en su Drive: {sum(f.stat().st_size for f in ARCH.values() if f) / 1e6:.1f} MB")

meta = {"estudiante": ESTUDIANTE, "zona": {k: v for k, v in Z.items() if k != "geom"}, "periodo": [ANIO_INI, ANIO_FIN],
        "modo_datos": MODO_DATOS, "celdas_era5": len(CELDAS), "recuadro_era5": AREA, "recuadro_core": AREA_CORE,
        "dia": {"ERA5": f"hora local desde las {INICIO_DIA if MODO_DATOS == 'propio' else 0:02d}:00",
                "CORe": "01:00–01:00 hora local"},
        "fecha": datetime.date.today().isoformat(),
        "fuentes": ({"era5": f"{era5.SERIES} (CDS, cuenta personal)", "era5_terreno": era5.MENSUAL,
                     "core": f"{core.BASE_6H} (flujos de 6 h)" if CORE_DIARIO else f"{core.BASE_MES} (promedios mensuales)"}
                    if MODO_DATOS == "propio"
                    else {"release": f"{release.REPO} · {release.ETIQUETA}"}),
        "archivos": {k: (None if v is None else v.name) for k, v in ARCH.items()}}
(DATOS / "inventario.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
INVENTARIO.to_csv(DATOS / "inventario.csv", index=False, encoding="utf-8")

In [ ]:
# Vista rápida: media del periodo, con el límite de la zona
E = xr.open_dataset(ARCH["era5_mensual"])
vistas = [("ERA5 · T media (°C)", E["tmed"].mean("time")), ("ERA5 · P (mm/mes)", E["pr"].mean("time"))]
if ARCH.get("core_mensual"):
    vistas.append(("CORe · T media (°C)", xr.open_dataset(ARCH["core_mensual"])["tmed"].mean("time")))
fig, ax = plt.subplots(1, len(vistas), figsize=(4.5 * len(vistas), 4))
for a, (t, da) in zip(np.atleast_1d(ax), vistas):
    da.plot(ax=a, cmap="RdYlBu_r" if "T" in t else "YlGnBu"); gpd.GeoSeries([GEOM]).boundary.plot(ax=a, color="k", lw=0.8); a.set_title(t)
plt.tight_layout(); plt.show()

## 9. Antes de seguir

- ¿Todos los archivos tienen estado `ok`? Con `CORE_DIARIO = False`, `core_diario` no aplica. Si falta otro, vuelva a ejecutar la celda de descarga; si persiste, avise al docente. Puede seguir con lo que tenga.
- ¿Cuántas celdas de cada reanálisis cubren su zona? Esa diferencia de resolución pesará en todo lo que sigue.
- Opción A: puede borrar las carpetas `_celdas_era5` y `_dias_core` cuando termine la práctica; si las deja, una nueva descarga de la misma zona y periodo será inmediata.

**Siguiente:** `CR02_series_zona` (las dos carreras).

**Citas.** Hersbach, H. et al. (2020). The ERA5 global reanalysis. *Q. J. R. Meteorol. Soc.*, 146, 1999–2049 · Copernicus Climate Change Service. *ERA5 hourly time-series data on single levels from 1940 to present*. C3S CDS · Hersbach, H. et al. (2023). *ERA5 monthly averaged data on single levels from 1940 to present*. C3S CDS. DOI 10.24381/cds.f17050d7 · NOAA/NCEP Climate Prediction Center. CORe, Conventional Observation Reanalysis.